In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import yaml
import lightning_scripts.lightning_classifier_matched_speech_in_noise as lightning 
import importlib
import torch
import pandas as pd 

In [2]:
### Reduce precision for notebook dev 
torch.set_float32_matmul_precision = 'medium'

# Min example of training loop in notebook

#### The following scripts are used under the hood:
* Audio transforms: `robustness/audio_functions/audio_transforms.py` 
* Architecture wrapper: `lightning_scripts/architectures.py`
* Dataset Class: `lightning_scripts/jsinV3DataLoader_precombined_batched.py`
* Lightning module: `lightning_scripts/lightning_classifier_matched_speech_in_noise.py`
* Yaml config: `model_configs/word_speaker_audioset_resnet18_MatchedSpeechInNoiseDatasetBatched.yaml`


___
### Start here

In [3]:
####### Get config 


config_path = "model_configs/word_speaker_audioset_resnet18_MatchedSpeechInNoiseDatasetBatched.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 32 ## EG to change batch size 
config['hparas']['lr'] = 1e-1 ## Eg to change LR  


In [ ]:
### Init and run min training loop
importlib.reload(lightning)
LitWordAudioSetModel = lightning.LitWordAudioSetModel

model = LitWordAudioSetModel(config)

# If desired, easy way to look at gradients - need to login with existing account 
# from lightning.pytorch.loggers import WandbLogger
# wandb_logger = WandbLogger(project="dev_matched_supervised")
# wandb_logger.watch(model.model, log="all", log_freq=5)

#############################################
# Commented flags for useful knobs to tweak
#############################################
trainer = L.Trainer(
    # limit_train_batches=5,
    # limit_val_batches=0,
    max_epochs=config['hparas']['epochs'],
    # strategy='ddp_notebook',
    # gradient_clip_val=1,
    # overfit_batches=.001,
    # logger=wandb_logger,
    devices=1)

trainer.fit(model)

### Current Dataset class

Copied below for quick reference. This lives in `lightning_scripts/jsinV3DataLoader_precombined_batched.py` starting at line 331. 

```python
from lightning_scripts import jsinV3DataLoader_precombined_batched 
dataset = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched(**kwargs)
```


In [ ]:
import h5py
import torch
import glob
import pickle
import numpy as np
from robustness.audio_functions import audio_transforms
import pandas as pd

class MatchedSpeechInNoiseDatasetBatched(torch.utils.data.Dataset):
    def __init__(
            self,
            speech_h5_path,
            noise_h5_path,
            low_db=-10,
            high_db=10,
            db_spl=60,
            batch_size=1,
            transform=None,
            target_keys=None,
            overfit=False,
    ):
        super().__init__()
        self.speech_files = h5py.File(speech_h5_path, 'r', swmr=True)
        self.noise_files = h5py.File(noise_h5_path, 'r', swmr=True)
        self.speech_metadata = pd.read_hdf(speech_h5_path)
        self.speech_metadata = self.speech_metadata.dropna() ## Removes null label 

        self.noise_metadata = pd.read_hdf(noise_h5_path)
        self.noise_metadata = self.noise_metadata.dropna() ## Removes null label 

        self.num_noise_files = len(self.noise_metadata)
        self.batch_size = batch_size
        self.target_keys = target_keys
    
        self.random_crop = audio_transforms.RandomCrop(40000)
        self.matched_random_crop = audio_transforms.MatchedRandomSignalCrops(40000)
        self.matched_combiner = audio_transforms.MatchedCombineWithRandomDBSNR(low_db, high_db)
        self.set_dbSPL = audio_transforms.DBSPLNormalizeForegroundAndBackground(db_spl)

    def class_map(self):
        """
        Loads the mapping between the word IDX and human readable word map.
        """
        word_and_speaker_encodings = pickle.load(
            open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
        )
        class_map = word_and_speaker_encodings["word_idx_to_word"]
        return class_map, word_and_speaker_encodings

    def __len__(self):
        return len(self.speech_metadata) // (2 * self.batch_size)
    
    def __getitem__(self, idx):
        speech = self.speech_files['ndarray_data']['signal'][idx:idx + self.batch_size * 2]
        noise_idx = np.random.randint(self.num_noise_files - self.batch_size * 2)
        noise = self.noise_files['ndarray_data']['signal'][noise_idx:noise_idx + self.batch_size * 2]
        
        # shuffle the indices of speech and noise - externalize for label ix-ing
        speech_ixs = np.random.permutation(speech.shape[0])
        noise_ixs = np.random.permutation(noise.shape[0])
        
        output_11, output_12, output_21, output_22 = [], [], [], []

        if self.target_keys:
            # track labels of each combo 
            target_11 = {}
            target_12 = {}
            target_21 = {}
            target_22 = {}
            
            for target_key in self.target_keys:
                for target_set in [target_11, target_12, target_21, target_22]:
                    target_set[target_key] = []
                
        for i in range(self.batch_size):
            # map batch ix to shuffle ix to make label alignment easy
            speech_1_ix, speech_2_ix = speech_ixs[i * 2], speech_ixs[i * 2 + 1]
            noise_1_ix, noise_2_ix = noise_ixs[i * 2] , noise_ixs[i * 2 + 1]
            # get speech example
            speech_1, speech_2 = speech[speech_1_ix], speech[speech_2_ix]
            if len(speech_1) > len(speech_2):
                speech_1, speech_2 = speech_2, speech_1
                # track ix swap for labeling 
                speech_1_ix, speech_2_ix = speech_2_ix, speech_1_ix

            # get noise example 
            noise_1, noise_2 = self.random_crop(noise[noise_1_ix]), self.random_crop(noise[noise_2_ix])
            noise_1, noise_2 = torch.tensor(noise_1), torch.tensor(noise_2)

            # store labels if supervised
            if self.target_keys:
                for target_key in self.target_keys:
                    target_type, target_name = target_key.split("/")
                    if target_type == 'signal':
                        target_11[target_key].append(self.speech_metadata.loc[speech_1_ix, target_name].item())
                        target_12[target_key].append(self.speech_metadata.loc[speech_1_ix, target_name].item())
                        target_21[target_key].append(self.speech_metadata.loc[speech_2_ix, target_name].item())
                        target_22[target_key].append(self.speech_metadata.loc[speech_2_ix, target_name].item())
                    elif target_type == 'noise':
                        target_11[target_key].append(self.noise_files['ndarray_data']['labels_binary_via_int'][noise_1_ix])
                        target_12[target_key].append(self.noise_files['ndarray_data']['labels_binary_via_int'][noise_2_ix])
                        target_21[target_key].append(self.noise_files['ndarray_data']['labels_binary_via_int'][noise_1_ix])
                        target_22[target_key].append(self.noise_files['ndarray_data']['labels_binary_via_int'][noise_2_ix])
            
            # randomly crop clips to be the same length (2 seconds = 40000 samples)
            cropped_11, cropped_21 = self.matched_random_crop(speech_1, speech_2)
            cropped_12, cropped_22 = self.matched_random_crop(speech_1, speech_2)

            cropped_11, cropped_21 = torch.tensor(cropped_11), torch.tensor(cropped_21)
            cropped_12, cropped_22 = torch.tensor(cropped_12), torch.tensor(cropped_22)

            # randomly mix the speech and noise with the same DBSNR
            combined_11, combined_21 = self.matched_combiner(cropped_11, cropped_21, noise_1, noise_1)
            combined_12, combined_22 = self.matched_combiner(cropped_12, cropped_22, noise_2, noise_2)  

            # set dB SPL for mixtures 
            combined_11, _ = self.set_dbSPL(combined_11, None)
            combined_12, _ = self.set_dbSPL(combined_12, None)
            combined_21, _ = self.set_dbSPL(combined_21, None)
            combined_22, _ = self.set_dbSPL(combined_22, None)

            output_11.append(combined_11)
            output_12.append(combined_12)
            output_21.append(combined_21)
            output_22.append(combined_22)
        
        output_11 = torch.stack(output_11).float()
        output_12 = torch.stack(output_12).float()
        output_21 = torch.stack(output_21).float()
        output_22 = torch.stack(output_22).float()
        
        # format targets 
        for target in [target_11, target_12, target_21 , target_22]:
            for target_key, target_list in target.items():
                if 'noise' in target_key:
                    target[target_key] = torch.from_numpy(np.stack(target_list, axis=0)).float()
                else:
                    target[target_key] = torch.tensor(target_list)
    
        if self.target_keys:
            return [output_11, output_12, output_21, output_22], [target_11, target_12, target_21 , target_22]

        return output_11, output_12, output_21, output_22